In [ ]:
!nvidia-smi -L
!pip -q install PyWavelets torchaudio soundfile kagglehub scikit-learn --upgrade

In [ ]:
REPO = "https://github.com/YOUR_GITHUB_USERNAME/sigwavnet-music-genre.git"
import os, sys, subprocess, numpy as np, torch, json, time
if not os.path.isdir("/content/sigwavnet-music-genre"):
    subprocess.run(["git", "clone", REPO, "/content/sigwavnet-music-genre"])
sys.path.insert(0, "/content/sigwavnet-music-genre/src")
os.chdir("/content/sigwavnet-music-genre")

from data import (index_gtzan, index_fma, build_cache, split_by_group,
                  SegmentDataset, class_weights, GTZAN_GENRES)
from train import build_model, train_model
from evaluate import (evaluate_clips, ensemble_clips, plot_confusion,
                      plot_history, plot_kernels, save_results)

In [ ]:
import kagglehub
kagglehub.login()
GTZAN_ROOT = os.path.join(
    kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification"), "Data")

gtz = build_cache(index_gtzan(GTZAN_ROOT), "/content/cache/gtzan")
gtz_tr, gtz_va, gtz_te = split_by_group(gtz, test_size=0.15, val_size=0.15, seed=42)
print(f"clips {len(gtz_tr)}/{len(gtz_va)}/{len(gtz_te)}")

CFG = dict(level=6, hidden_dim=64, n_layers=3, n_channel=64,
           kernel="db10", mode="PerFilter", initHT=1.0, alpha=10.0, dropout=0.1)
SEG_SEC, VAL_HOP, TEST_HOP, REPEATS = 3.0, 3.0, 1.0, 10

In [ ]:
tr_ds = SegmentDataset(gtz_tr, GTZAN_GENRES, SEG_SEC, train=True,
                       repeats=REPEATS, speed_perturb=0.5)
va_ds = SegmentDataset(gtz_va, GTZAN_GENRES, SEG_SEC, VAL_HOP,  train=False)
te_ds = SegmentDataset(gtz_te, GTZAN_GENRES, SEG_SEC, TEST_HOP, train=False)
print(f"{len(tr_ds)} train segments/epoch | {len(te_ds)} test windows "
      f"({len(te_ds)/len(gtz_te):.0f} per track)")

EPOCHS = 90        # val F1 was still climbing at 40

In [ ]:
t0 = time.time()
m1, hist1 = train_model(
    build_model(n_output=10, **CFG), tr_ds, va_ds, classes=GTZAN_GENRES,
    weights=None, epochs=EPOCHS, batch_size=32,
    lr_head=2e-3, lr_encoder=1e-3, lr_wavelet=5e-4,
    gamma=0.0, mixup_alpha=0.15, label_smoothing=0.0,
    ckpt_path="push_seed0.pt", seed=0)
print("took", round((time.time()-t0)/60, 1), "min")

r1 = evaluate_clips(m1, te_ds, GTZAN_GENRES)
print(f"\nT1 single model -> clip acc {r1['clip_acc']:.4f}  F1 {r1['clip_f1']:.4f}")
print(r1["report"])
save_results("tier1_seed0", r1)

In [ ]:
models = [m1]
for sd in (1, 2):
    print(f"\n########## seed {sd}")
    m, _ = train_model(
        build_model(n_output=10, **CFG), tr_ds, va_ds, classes=GTZAN_GENRES,
        weights=None, epochs=EPOCHS, batch_size=32,
        lr_head=2e-3, lr_encoder=1e-3, lr_wavelet=5e-4,
        gamma=0.0, mixup_alpha=0.15, label_smoothing=0.0,
        ckpt_path=f"push_seed{sd}.pt", seed=sd, log_every=0)
    models.append(m)

re_ = ensemble_clips(models, te_ds, GTZAN_GENRES)
print(f"\n3-seed ensemble -> clip acc {re_['clip_acc']:.4f}  F1 {re_['clip_f1']:.4f}")
print(re_["report"])
save_results("tier2_ensemble3", re_)
plot_confusion(re_["cm"], GTZAN_GENRES, "confusion_ensemble.png",
               title="GTZAN test — 3-seed ensemble")

In [ ]:
FMA_DIR = "/content/fma"
if not os.path.isdir(f"{FMA_DIR}/fma_small"):
    !mkdir -p {FMA_DIR}
    !wget -q --show-progress https://os.unil.cloud.switch.ch/fma/fma_metadata.zip -O {FMA_DIR}/meta.zip
    !wget -q --show-progress https://os.unil.cloud.switch.ch/fma/fma_small.zip    -O {FMA_DIR}/small.zip
    !cd {FMA_DIR} && unzip -q meta.zip && unzip -q small.zip
!ls {FMA_DIR}

In [ ]:
fma = index_fma(f"{FMA_DIR}/fma_small", f"{FMA_DIR}/fma_metadata/tracks.csv", subset="small")
fma = build_cache(fma, "/content/cache/fma", max_seconds=10)   # centre 10 s only
FMA_CLASSES = sorted(fma.label.unique())
print(len(fma), FMA_CLASSES)

fma_tr, fma_va, _ = split_by_group(fma, test_size=0.05, val_size=0.10, seed=42)
fma_tr_ds = SegmentDataset(fma_tr, FMA_CLASSES, SEG_SEC, train=True,
                           repeats=3, speed_perturb=0.5)
fma_va_ds = SegmentDataset(fma_va, FMA_CLASSES, SEG_SEC, 3.0, train=False)
print(f"{len(fma_tr_ds)} FMA train segments/epoch")

In [ ]:
mp, hist_p = train_model(
    build_model(n_output=len(FMA_CLASSES), **CFG), fma_tr_ds, fma_va_ds,
    classes=FMA_CLASSES, weights=class_weights(fma_tr, FMA_CLASSES),
    epochs=25, batch_size=32,
    lr_head=2e-3, lr_encoder=1e-3, lr_wavelet=5e-4,
    gamma=0.0, mixup_alpha=0.15, label_smoothing=0.0,
    ckpt_path="fma_pretrain.pt", log_every=200)

In [ ]:
ft_models = []
for sd in (0, 1):
    mb = build_model(n_output=len(FMA_CLASSES), **CFG)
    mb.load_pretrained("fma_pretrain.pt")
    mb.replace_head(10)
    mb, _ = train_model(mb, tr_ds, va_ds, classes=GTZAN_GENRES,
        weights=None, epochs=EPOCHS, batch_size=32,
        lr_head=2e-3, lr_encoder=1e-3, lr_wavelet=3e-4,
        gamma=0.0, mixup_alpha=0.15, label_smoothing=0.0,
        freeze_epochs=1, ckpt_path=f"push_fma_seed{sd}.pt", seed=sd, log_every=0)
    ft_models.append(mb)

r3 = ensemble_clips(ft_models, te_ds, GTZAN_GENRES)
print(f"\nFMA-pretrained ensemble -> clip acc {r3['clip_acc']:.4f}  F1 {r3['clip_f1']:.4f}")
print(r3["report"])
save_results("tier3_fma_ensemble", r3)

In [ ]:
allm = models + ft_models
rall = ensemble_clips(allm, te_ds, GTZAN_GENRES)
print(f"all-{len(allm)} ensemble -> clip acc {rall['clip_acc']:.4f}  F1 {rall['clip_f1']:.4f}")
save_results("final_ensemble", rall)

plot_confusion(rall["cm"], GTZAN_GENRES, "confusion_final.png",
               title="GTZAN test — final ensemble")
plot_kernels(ft_models[0], "learned_kernels_final.png", n_levels=3)

for k, v in json.load(open("results.json")).items():
    print(f"{k:<24} clip acc {v['clip_acc']:.4f}  F1 {v['clip_f1']:.4f}")

In [ ]:
!zip -q -r push_artefacts.zip results.json *.png *_history.json push_*.pt fma_pretrain.pt
from google.colab import files; files.download("push_artefacts.zip")